# Paired M0 versus M1 weather experiment: development and model lock

This notebook prepares the controlled weather ablation for RQ3 without evaluating the outer test. The paired population contains only rows with complete M1 weather, and the same row IDs, households, dates, labels, and internal split are used for M0 and M1. The selected family and hyperparameters from Notebook 05 are held fixed within each target; M1 adds only the daily weather features.

Both versions fit on paired `model_development` rows, calibrate on paired `calibration` rows, and choose F1 thresholds on paired `threshold_selection` rows using the same policy as Notebook 06. Threshold results are selection diagnostics, not independent performance estimates. This notebook saves paired model bundles and a frozen experiment manifest for the later final evaluation. It does not load or score outer-test labels.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (average_precision_score, brier_score_loss, f1_score,
                             log_loss, precision_score, recall_score, roc_auc_score)
from sklearn.calibration import calibration_curve
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

ROOT=Path.cwd().resolve()
while not (ROOT/"data"/"processed").exists() and ROOT!=ROOT.parent: ROOT=ROOT.parent
if not (ROOT/"data"/"processed").exists(): raise RuntimeError("Could not locate project root")
DATA_DIR=ROOT/"data"/"processed"
TABLE_DIR=ROOT/"reports"/"tables"/"08_weather_model_comparison"
FIGURE_DIR=ROOT/"reports"/"figures"/"08_weather_model_comparison"
MODEL_DIR=ROOT/"models"/"08_weather_model_comparison"
for d in (TABLE_DIR,FIGURE_DIR,MODEL_DIR): d.mkdir(parents=True,exist_ok=True)

PATHS={
 "P95_M0":DATA_DIR/"05_model_ready_m0_p95.csv", "P90_M0":DATA_DIR/"05_model_ready_m0_p90.csv",
 "P95_M1":DATA_DIR/"07_model_ready_m1_p95.csv", "P90_M1":DATA_DIR/"07_model_ready_m1_p90.csv",
}
MASTER_PATH=DATA_DIR/"04_master_split.csv"
PAIR_AUDIT_PATH=DATA_DIR/"07_m0_m1_pairing_audit.csv"
WEATHER_MANIFEST_PATH=ROOT/"data"/"external"/"weather"/"07_fixed_run_weather_manifest.json"
SELECTED_PATH=ROOT/"reports"/"tables"/"05_model_training_tuning"/"05_selected_models.json"
TARGET_CONFIG_PATH=ROOT/"reports"/"tables"/"03a_target_design_experiments"/"03a_selected_target_config.json"
FEATURE_QUALITY_CONFIG_PATH=ROOT/"reports"/"tables"/"03b_historical_feature_quality_experiments"/"03b_selected_feature_quality_config.json"
RANDOM_STATE=42
THRESHOLD_GRID=np.round(np.arange(0.01,1.00,0.01),2)
INTERNAL_STAGES=("model_development","calibration","threshold_selection")

def read_json(path): return json.loads(Path(path).read_text(encoding="utf-8"))
def sha256(path):
 h=hashlib.sha256()
 with Path(path).open("rb") as f:
  for block in iter(lambda:f.read(8*1024*1024),b""): h.update(block)
 return h.hexdigest()
def numeric_x(frame,features):
 return frame[features].apply(lambda c:pd.to_numeric(c,errors="coerce").astype("float64"))
def probability_logit(probability):
 p=np.clip(np.asarray(probability,dtype=float),1e-6,1-1e-6)
 return np.log(p/(1-p)).reshape(-1,1)


## 1. Build the complete-weather paired cohort and parse internal rows only

The pairing audit and frozen master split define membership before model-ready feature and target data are parsed. The selective loader scans only `row_id` across the source CSV, then parses only paired rows assigned to the three internal development blocks. Outer-test feature and target rows are skipped.

In [ ]:
for path in [*PATHS.values(),MASTER_PATH,PAIR_AUDIT_PATH,WEATHER_MANIFEST_PATH,SELECTED_PATH]:
 if not path.exists(): raise FileNotFoundError(f"Missing prerequisite: {path}")
master=pd.read_csv(MASTER_PATH,dtype={"row_id":"string","household_id":"string"})
master["target_date"]=pd.to_datetime(master["target_date"],errors="raise")
pair_audit=pd.read_csv(PAIR_AUDIT_PATH,dtype={"row_id":"string","household_id":"string"})
selected=read_json(SELECTED_PATH); weather_manifest=read_json(WEATHER_MANIFEST_PATH)
target_config=read_json(TARGET_CONFIG_PATH); feature_quality_config=read_json(FEATURE_QUALITY_CONFIG_PATH)
if target_config.get("status")!="frozen_primary_research_policy" or feature_quality_config.get("status")!="frozen_feature_quality_policy": raise ValueError("Frozen policy provenance is required")
if master.row_id.isna().any() or master.row_id.duplicated().any(): raise ValueError("Master row IDs must be unique")
if pair_audit.row_id.isna().any() or pair_audit.row_id.duplicated().any(): raise ValueError("Pair-audit row IDs must be unique")
if not weather_manifest.get("validation_passed",False): raise ValueError("Notebook 07 validation did not pass")
if set(selected)!={"P95","P90"}: raise ValueError("Notebook 05 must provide P95 and P90 model specifications")
if not set(INTERNAL_STAGES).issubset(set(master.internal_split.dropna().unique())): raise ValueError("Frozen internal split blocks missing")

complete_ids=set(pair_audit.loc[pair_audit.m1_weather_complete.fillna(False).astype(bool),"row_id"].astype(str))
paired_internal_ids=set(master.loc[master.row_id.astype(str).isin(complete_ids)&master.internal_split.isin(INTERNAL_STAGES),"row_id"].astype(str))
if not paired_internal_ids: raise ValueError("No paired complete-weather internal rows")

def read_selected_rows(path,expected_ids):
 row_ids=pd.read_csv(path,usecols=["row_id"],dtype={"row_id":"string"}).row_id
 positions=np.flatnonzero(row_ids.isin(expected_ids).to_numpy())
 found=set(row_ids.iloc[positions].astype(str))
 if found!=expected_ids: raise ValueError(f"{path.name}: row ID mismatch missing={len(expected_ids-found)}, unexpected={len(found-expected_ids)}")
 keep=set(positions.tolist())
 frame=pd.read_csv(path,dtype={"row_id":"string","household_id":"string"},skiprows=lambda line:line>0 and (line-1) not in keep)
 if set(frame.row_id.astype(str))!=expected_ids or frame.row_id.duplicated().any(): raise ValueError(f"Selective load mismatch for {path.name}")
 return frame

frames={name:read_selected_rows(path,paired_internal_ids) for name,path in PATHS.items()}
assign=master[["row_id","household_id","target_date","outer_split","internal_split","seen_in_training","test_cohort"]]
for name,frame in frames.items():
 frame["measurement_date"]=pd.to_datetime(frame["measurement_date"],errors="raise")
 frame=frame.merge(assign,on="row_id",how="left",validate="one_to_one",suffixes=("","_master"))
 if frame.internal_split.isna().any() or frame.outer_split.eq("test").any(): raise ValueError(f"{name}: non-internal or test rows were parsed")
 if not frame.measurement_date.eq(frame.target_date).all(): raise ValueError(f"{name}: date does not match the frozen split")
 frames[name]=frame

# Every target and dataset in the ablation must contain the exact same paired observations.
for target in ("P95","P90"):
 m0=frames[f"{target}_M0"]; m1=frames[f"{target}_M1"]
 for col in ("row_id","household_id","measurement_date","target","outer_split","internal_split","test_cohort"):
  if not m0[col].reset_index(drop=True).equals(m1[col].reset_index(drop=True)): raise ValueError(f"{target}: M0/M1 mismatch in {col}")
 if not m0.row_id.reset_index(drop=True).equals(frames["P95_M0"].row_id.reset_index(drop=True)): raise ValueError(f"{target}: cross-target paired row IDs differ")
 if not set(m0.internal_split.unique())==set(INTERNAL_STAGES): raise ValueError(f"{target}: one or more internal blocks missing")

WEATHER_FEATURES=weather_manifest["daily_weather_features"]
if "shortwave_radiation_mean_wm2" in WEATHER_FEATURES or "shortwave_radiation_sum_kwh_m2" not in WEATHER_FEATURES: raise ValueError("Notebook 07 weather feature schema does not follow the frozen radiation policy")
PREDICTORS={}
for target in ("P95","P90"):
 base=list(selected[target]["feature_list"]); augmented=base+WEATHER_FEATURES
 if len(augmented)!=len(set(augmented)): raise ValueError(f"{target}: duplicate M1 feature names")
 for name in (f"{target}_M0",f"{target}_M1"):
  missing=sorted(set(base if name.endswith("M0") else augmented)-set(frames[name].columns))
  if missing: raise ValueError(f"{name}: missing features {missing}")
 PREDICTORS[f"{target}_M0"]=base; PREDICTORS[f"{target}_M1"]=augmented

cohort_summary=pd.DataFrame([{"target":t,"internal_split":stage,"paired_rows":int((frames[f"{t}_M0"].internal_split==stage).sum()),
                              "positive_events":int(frames[f"{t}_M0"].loc[frames[f"{t}_M0"].internal_split.eq(stage),"target"].sum()),
                              "start_date":frames[f"{t}_M0"].loc[frames[f"{t}_M0"].internal_split.eq(stage),"measurement_date"].min(),
                              "end_date":frames[f"{t}_M0"].loc[frames[f"{t}_M0"].internal_split.eq(stage),"measurement_date"].max()}
                             for t in ("P95","P90") for stage in INTERNAL_STAGES])
cohort_summary.to_csv(TABLE_DIR/"08_paired_internal_cohort.csv",index=False)
display(cohort_summary)
print("Paired internal rows parsed:",f"{len(paired_internal_ids):,}","| outer-test rows parsed: 0")


## 2. Fit matched models, calibrators, and operating thresholds

For each target, M0 and M1 use the same selected family, target-specific parameters, model-development rows, calibration rows, threshold-selection rows, random seed, and threshold rule. Only the M1 feature matrix adds the eight complete-weather summaries. Class-ratio weighting, where selected, is calculated from paired model-development labels.

In [ ]:
def build_estimator(family,params,y_fit):
 params=dict(params)
 if family=="XGBoost":
  mode=params.pop("scale_pos_weight_mode","one"); ratio=float((y_fit==0).sum()/max((y_fit==1).sum(),1))
  scales={"one":1.0,"sqrt_ratio":ratio**0.5,"half_ratio":ratio/2.0,"full_ratio":ratio}
  if mode not in scales: raise ValueError(f"Unknown XGBoost weighting mode: {mode}")
  estimator=XGBClassifier(**params,scale_pos_weight=scales[mode],eval_metric="logloss",tree_method="hist",n_jobs=2,random_state=42,verbosity=0)
  return Pipeline([("imputer",SimpleImputer(strategy="median",add_indicator=True,keep_empty_features=True)),("model",estimator)])
 if family=="Random Forest":
  estimator=RandomForestClassifier(**params,n_jobs=2,random_state=42)
  return Pipeline([("imputer",SimpleImputer(strategy="median",add_indicator=True,keep_empty_features=True)),("model",estimator)])
 if family=="Logistic Regression":
  estimator=LogisticRegression(**params,solver="lbfgs",max_iter=2000,random_state=42)
  return Pipeline([("imputer",SimpleImputer(strategy="median",add_indicator=True,keep_empty_features=True)),("model",estimator)])
 if family=="CatBoost":
  return CatBoostClassifier(**params,loss_function="Logloss",random_seed=42,verbose=False,allow_writing_files=False,thread_count=2)
 raise ValueError(f"Unsupported selected family: {family}")

def find_threshold(y,p,target,dataset):
 rows=[]
 for threshold in THRESHOLD_GRID:
  pred=(p>=threshold).astype("int8")
  rows.append({"target":target,"dataset":dataset,"threshold":float(threshold),"precision":precision_score(y,pred,zero_division=0),
               "recall":recall_score(y,pred,zero_division=0),"f1":f1_score(y,pred,zero_division=0)})
 curve=pd.DataFrame(rows)
 best=curve.sort_values(["f1","precision","threshold"],ascending=[False,False,False],kind="stable").iloc[0]
 return curve,best

models={}; calibrators={}; probability_sets={}; bundles={}; quality_rows=[]; threshold_rows=[]; curves=[]; stage_counts=[]
for target in ("P95","P90"):
 spec=selected[target]; family=spec["model_family"]; params=spec["parameters"]
 for dataset in ("M0","M1"):
  key=f"{target}_{dataset}"; frame=frames[key]; features=PREDICTORS[key]
  fit=frame.loc[frame.internal_split.eq("model_development")]; cal=frame.loc[frame.internal_split.eq("calibration")]; th=frame.loc[frame.internal_split.eq("threshold_selection")]
  yfit=fit.target.astype("int8").to_numpy(); ycal=cal.target.astype("int8").to_numpy(); yth=th.target.astype("int8").to_numpy()
  model=build_estimator(family,params,yfit)
  model.fit(numeric_x(fit,features),yfit)
  pcal=model.predict_proba(numeric_x(cal,features))[:,1]
  pth=model.predict_proba(numeric_x(th,features))[:,1]
  calibrator=LogisticRegression(C=1.0,solver="lbfgs",max_iter=2000,random_state=42)
  calibrator.fit(probability_logit(pcal),ycal)
  cal_calibrated=calibrator.predict_proba(probability_logit(pcal))[:,1]
  th_calibrated=calibrator.predict_proba(probability_logit(pth))[:,1]
  models[key]=model; calibrators[key]=calibrator
  probability_sets[key]={"calibration_raw":pcal,"calibration_calibrated":cal_calibrated,"threshold_raw":pth,"threshold_calibrated":th_calibrated}
  curve,best=find_threshold(yth,th_calibrated,target,dataset); curves.append(curve)
  threshold_rows.append({"target":target,"dataset":dataset,"model_family":family,"threshold":float(best.threshold),"precision":float(best.precision),"recall":float(best.recall),"f1":float(best.f1),
                        "threshold_rule":"maximize F1; ties by precision then higher threshold","selection_split":"threshold_selection","selection_rows":len(th),"selection_events":int(yth.sum())})
  stage_counts.append({"target":target,"dataset":dataset,"model_family":family,"parameters":json.dumps(params,sort_keys=True),"fit_rows":len(fit),"calibration_rows":len(cal),"threshold_rows":len(th)})
  for stage,y,raw,calprob,interpretation in (("calibration",ycal,pcal,cal_calibrated,"in-sample calibrator-fit diagnostic"),("threshold_selection",yth,pth,th_calibrated,"held apart from calibrator fit; threshold-selection data")):
   for probability_name,prob in (("raw",raw),("platt_calibrated",calprob)):
    quality_rows.append({"target":target,"dataset":dataset,"stage":stage,"probability":probability_name,"rows":len(y),"events":int(y.sum()),
                         "average_precision":average_precision_score(y,prob),"roc_auc":roc_auc_score(y,prob),"brier_score":brier_score_loss(y,prob),
                         "log_loss":log_loss(y,prob,labels=[0,1]),"interpretation":interpretation})
  bundles[key]={"target":target,"dataset":dataset,"model_role":"controlled_weather_ablation","research_questions":["RQ3"],"model_name":f"paired_weather_{target}_{dataset}",
                 "target_configuration_id":target_config["configuration_id"],"feature_quality_configuration_id":feature_quality_config["configuration_id"],
                 "training_population":"complete_weather_paired_development_rows","model_family":family,"selected_parameters":params,"features":features,
                "base_estimator":model,"platt_calibrator":calibrator,"threshold":float(best.threshold),
                "threshold_rule":"maximize F1 on threshold_selection; ties by precision then higher threshold",
                "fit_split":"model_development","calibration_split":"calibration","threshold_split":"threshold_selection",
                "paired_population":"m1_weather_complete rows only","outer_test_used":False}
  print(f"{key}: {family}; fit={len(fit):,}, calibration={len(cal):,}, threshold rows={len(th):,}, threshold={best.threshold:.2f}")

quality=pd.DataFrame(quality_rows); threshold_table=pd.DataFrame(threshold_rows); curves=pd.concat(curves,ignore_index=True); stage_counts=pd.DataFrame(stage_counts)
quality.to_csv(TABLE_DIR/"08_internal_probability_metrics.csv",index=False)
threshold_table.to_csv(TABLE_DIR/"08_paired_thresholds.csv",index=False)
curves.to_csv(TABLE_DIR/"08_threshold_search_internal.csv",index=False)
stage_counts.to_csv(TABLE_DIR/"08_model_fit_summary.csv",index=False)
display(threshold_table); display(quality)


## 3. Compare development-only calibration diagnostics

Reliability curves use `threshold_selection`, which did not fit either calibrator. They are descriptive diagnostics on the operating-point selection period. No outer-test score is calculated.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,5))
for ax,target in zip(axes,("P95","P90")):
 for dataset in ("M0","M1"):
  frame=frames[f"{target}_{dataset}"]; y=frame.loc[frame.internal_split.eq("threshold_selection"),"target"].astype("int8").to_numpy()
  prob=probability_sets[f"{target}_{dataset}"]["threshold_calibrated"]
  frac,mean=calibration_curve(y,prob,n_bins=10,strategy="quantile")
  ax.plot(mean,frac,marker="o",label=f"{dataset} calibrated")
 ax.plot([0,1],[0,1],"--",color="gray"); ax.set(title=f"{target}: paired reliability",xlabel="Mean predicted probability",ylabel="Observed event rate"); ax.legend()
fig.tight_layout(); fig.savefig(FIGURE_DIR/"08_paired_reliability_internal.png",dpi=160,bbox_inches="tight"); plt.show()


## 4. Save paired model handoff and validate the lock

The four bundles are for the later final-evaluation stage. Each M0/M1 pair shares one target-specific model family and exact hyperparameter dictionary, while each fitted model and calibrator uses its own matching feature set. The reserved outer test is absent from all frames used here.

In [ ]:
for key,bundle in bundles.items(): joblib.dump(bundle,MODEL_DIR/f"{key.lower()}_paired_weather.joblib")

checks=[]
def add_check(name,passed,detail): checks.append({"check":name,"passed":bool(passed),"detail":str(detail)})
add_check("outer_test_rows_not_loaded",all(not frame.outer_split.eq("test").any() for frame in frames.values()),"Selective row parser loaded internal paired IDs only")
add_check("same_paired_row_ids_across_targets_and_datasets",len({tuple(frames[k].row_id.astype(str)) for k in frames})==1,len(paired_internal_ids))
add_check("same_target_labels_within_each_pair",all(frames[f"{t}_M0"].target.equals(frames[f"{t}_M1"].target) for t in ("P95","P90")),"P95 and P90 checked")
add_check("same_family_and_parameters_within_target",all(bundles[f"{t}_M0"]["model_family"]==bundles[f"{t}_M1"]["model_family"] and bundles[f"{t}_M0"]["selected_parameters"]==bundles[f"{t}_M1"]["selected_parameters"] for t in ("P95","P90")),{t:selected[t]["model_family"] for t in selected})
add_check("M1_adds_weather_features_only",all(PREDICTORS[f"{t}_M1"]==PREDICTORS[f"{t}_M0"]+WEATHER_FEATURES for t in ("P95","P90")),WEATHER_FEATURES)
add_check("thresholds_selected_on_later_internal_block",set(threshold_table.selection_split)=={"threshold_selection"},"F1-maximizing policy with deterministic tie break")
add_check("all_paired_model_bundles_saved",all((MODEL_DIR/f"{key.lower()}_paired_weather.joblib").exists() for key in bundles),MODEL_DIR)
roundtrip=True; roundtrip_summary={}
for key in bundles:
 bundle=joblib.load(MODEL_DIR/f"{key.lower()}_paired_weather.joblib"); sample=frames[key].loc[frames[key].internal_split.eq("threshold_selection")].head(10)
 raw=bundle["base_estimator"].predict_proba(numeric_x(sample,bundle["features"]))[:,1]
 calibrated=bundle["platt_calibrator"].predict_proba(probability_logit(raw))[:,1]
 valid=(len(calibrated)==len(sample) and np.isfinite(calibrated).all() and ((calibrated>=0)&(calibrated<=1)).all())
 roundtrip=roundtrip and bool(valid); roundtrip_summary[key]={"sample_rows":len(sample),"valid_probabilities":bool(valid)}
add_check("saved_bundles_reload_and_predict",roundtrip,roundtrip_summary)
validation=pd.DataFrame(checks); display(validation); validation.to_csv(TABLE_DIR/"08_final_validation.csv",index=False)
if not validation.passed.all(): raise AssertionError("Notebook 08 validation failed")

manifest={"created_utc":datetime.now(timezone.utc).isoformat(),"stage":"08_paired_weather_model_lock",
 "outer_test_labels_loaded":False,"outer_test_scored":False,"outer_test_used_for_any_modeling_decision":False,
 "paired_filter":"m1_weather_complete == True","internal_rows_parsed":len(paired_internal_ids),
 "same_rows_households_targets_dates_and_internal_splits":True,"weather_features":WEATHER_FEATURES,
 "calibration_protocol":"Platt sigmoid fit on calibration; F1 threshold selected on threshold_selection, ties by precision then higher threshold",
 "paired_model_role":"controlled_weather_ablation","target_configuration_id":target_config["configuration_id"],"feature_quality_configuration_id":feature_quality_config["configuration_id"],
 "models":{key:{"model_role":"controlled_weather_ablation","research_questions":["RQ3"],"model_name":bundles[key]["model_name"],"model_family":bundles[key]["model_family"],"parameters":bundles[key]["selected_parameters"],"threshold":bundles[key]["threshold"],"bundle":str(MODEL_DIR/f"{key.lower()}_paired_weather.joblib")} for key in bundles},
 "notebook_05_selection_sha256":sha256(SELECTED_PATH),"notebook_07_manifest_sha256":sha256(WEATHER_MANIFEST_PATH),
 "validation_passed":bool(validation.passed.all()),"checks":checks,
 "interpretation":"Calibration metrics on calibration are in-sample. Threshold-selection metrics and thresholds are development selection diagnostics, not independent outer-test performance."}
(TABLE_DIR/"08_paired_weather_experiment_manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")
print("PAIRED M0/M1 EXPERIMENT LOCKED — no outer-test data were parsed or scored.")


## Handoff

**PAIRED M0/M1 EXPERIMENT LOCKED.** The four frozen `controlled_weather_ablation` bundles (paired_weather_m0/m1), feature lists, calibrators, and thresholds are ready for the later final evaluation. The paired common-weather cohort, model settings, and internal-only validation are recorded in the manifest. Do not interpret calibration-fit or threshold-selection results as final test performance.